<div style="border-left:4px solid #818cf8;padding:2px 0 2px 16px;margin:6px 0 18px;"><div style="font:800 27px/1.15 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;letter-spacing:-0.02em;">NL2SQL <span style="font-weight:500;color:#818cf8;">Architectures</span></div><div style="font:400 15px/1.55 -apple-system,BlinkMacSystemFont,Segoe UI,Inter,sans-serif;color:#71717a;margin-top:5px;">Four designs for the same question, run and measured.</div></div>

[Setup](https://www.kaggle.com/code/kirazul/nl2sql-1-setup) &nbsp;|&nbsp; [Understanding](https://www.kaggle.com/code/kirazul/nl2sql-2-understanding) &nbsp;|&nbsp; **Architectures** &nbsp;|&nbsp; [Run All](https://www.kaggle.com/code/kirazul/nl2sql-4-run-all)

## 1. Setup

The code is cloned from GitHub. The database, the index and the two models are
read from [notebook 1](https://www.kaggle.com/code/kirazul/nl2sql-1-setup)'s saved output, where they already are. Nothing
is downloaded or rebuilt here.

Before running: **Add Input > Notebook Output > NL2SQL 1 Setup**, add the secrets
listed below under **Add-ons > Secrets**, and enable Internet.

In [ ]:
%%capture --no-stderr
!pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
    "llama-cpp-python>=0.3" "gliner2>=1.3" "langgraph>=1.0" "langsmith>=0.10" \
    "fastapi>=0.115" "uvicorn[standard]>=0.34" "pydantic-settings>=2.6" \
    "sqlglot>=25.0" "rapidfuzz>=3.10" "pyyaml>=6.0" "httpx>=0.27" "python-dotenv>=1.0"

In [ ]:
import os, re, sys, json, time, shutil, subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
WORK      = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
INPUTS    = Path("/kaggle/input")
REPO      = "https://github.com/Kirazul/NL2SQL-demo.git"

SECRETS = {
    "GITHUB_TOKEN":       "clone the code (the repository is private)",
    "GROQ_API_KEY":       "the cloud model that writes the SQL",
    "OPENROUTER_API_KEY": "fallback when Groq rate-limits",
    "LANGSMITH_API_KEY":  "tracing backend",
    "PUBLISH_TOKEN":      "announce this session to the web interface",
}
REQUIRED = ('GROQ_API_KEY',)


def secret(label, default=""):
    """One secret, by label. Kaggle grants access per notebook, not per account."""
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(label) or default
        except Exception:
            pass
    return os.environ.get(label, default)


def load_secrets(project=None):
    """Read every label into the environment and print what was found.

    An empty secret is removed rather than set blank, so the package falls back to
    its own default instead of an empty string.
    """
    local = {}
    if not ON_KAGGLE and project and (project / ".env").exists():
        for line in (project / ".env").read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                label, _, value = line.partition("=")
                local[label.strip()] = value.strip().strip("\"'")

    for label in SECRETS:
        value = secret(label) or local.get(label, "")
        if value:
            os.environ[label] = value
        else:
            os.environ.pop(label, None)

    for label, purpose in SECRETS.items():
        if os.environ.get(label):
            state = "ok"
        elif label in REQUIRED:
            state = "REQUIRED"
        else:
            state = "-"
        print(f"  {label:<20}{state:<10}{purpose}")

    absent = [l for l in REQUIRED if not os.environ.get(l)]
    if absent:
        where = "Add-ons > Secrets, in this notebook" if ON_KAGGLE else ".env"
        print(f"\n  Missing: {', '.join(absent)}. Set it in {where} and run this cell again.")
    return not absent


def get_code():
    """Clone the repository into a writable directory and put it on the path.

    Kaggle mounts every input read-only and notebook 1 writes a database next to
    the package, so the code never runs from where it is mounted.
    """
    if (Path.cwd() / "src/hybridsql").exists():
        return Path.cwd()

    target = WORK / "nl2sql"
    if (target / "src/hybridsql").exists():
        return target

    token = secret("GITHUB_TOKEN")
    url = REPO.replace("https://", f"https://{token}@") if token else REPO
    done = subprocess.run(["git", "clone", "--depth", "1", "--quiet", url, str(target)],
                          capture_output=True, text=True)
    if done.returncode:
        detail = done.stderr.replace(token, "***") if token else done.stderr
        raise SystemExit(
            "Could not clone the repository.\n"
            "  It is private, so GITHUB_TOKEN must be set under Add-ons > Secrets\n"
            "  and attached to this notebook.\n\n" + detail
        )
    return target
ARTEFACTS = {
    "database":    ("data/warehouse/eicu.db",                   None),
    "value index": ("data/warehouse/value_index.db",            None),
    "GLiNER2":     ("models/gliner2-base-v1",                   "model.safetensors"),
    "Qwen3-1.7B":  ("models/qwen3-1.7b/Qwen3-1.7B-Q4_K_M.gguf", None),
}
DEPTHS = ("", "*/", "*/*/", "*/*/*/", "*/*/*/*/", "*/*/*/*/*/")


def whole(path, probe=None):
    """Present and finished. A model directory with no weights in it is neither."""
    return (path / probe).exists() if probe else path.exists()


def find_input(relative, probe=None):
    """The first attached input carrying `relative`, at whatever depth it sits."""
    if not INPUTS.exists():
        return None
    for prefix in DEPTHS:
        for hit in sorted(INPUTS.glob(prefix + relative)):
            if whole(hit, probe):
                return hit
    return None


def attached_inputs():
    """The inputs actually attached, named by what they carry rather than by the
    directory level Kaggle happens to mount them under."""
    if not INPUTS.exists():
        return []
    markers = ("src", "data", "models", "nl2sql")
    return [p.relative_to(INPUTS).as_posix()
            for pattern in ("*", "*/*", "*/*/*")
            for p in sorted(INPUTS.glob(pattern))
            if p.is_dir() and any((p / m).exists() for m in markers)]


def locate(project):
    """Every artefact, in the working copy or in an attached input."""
    found, missing = {}, []
    for label, (relative, probe) in ARTEFACTS.items():
        if label == "value index":
            continue                       # always beside the database, see below
        local = project / relative
        path = local if whole(local, probe) else find_input(relative, probe)
        (found.__setitem__(label, path) if path else missing.append(label))

    # The package derives the index path from the database path, so the two must
    # be in the same directory. Looking for it anywhere else would resolve here
    # and fail there.
    if "database" in found:
        index = found["database"].with_name("value_index.db")
        found["value index"] = index if index.exists() else missing.append("value index")
    else:
        missing.append("value index")
    return found, [m for m in missing if m]


def configure(found):
    os.environ["DB_PATH"]             = str(found["database"])
    os.environ["GLINER_MODEL"]        = str(found["GLiNER2"])
    os.environ["LOCAL_LLM_GGUF_PATH"] = str(found["Qwen3-1.7B"])
    os.environ["LOCAL_LLM_THREADS"]   = str(max(2, os.cpu_count() or 4))
    os.environ["LOCAL_LLM_BACKEND"]   = "llamacpp"
    os.environ["PRIVACY_MODE"]        = "demo"
    os.environ["LANGSMITH_PROJECT"]   = "nl2sql"
    os.environ["LANGSMITH_TRACING"]   = "1" if os.environ.get("LANGSMITH_API_KEY") else "0"


def size_mb(path):
    if path.is_dir():
        return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e6
    return path.stat().st_size / 1e6 if path.exists() else 0.0


def show(found):
    for label, (relative, _) in ARTEFACTS.items():
        path = found.get(label)
        if path is None:
            print(f"  {label:<14}{'missing':>10}")
            continue
        root = path.parents[len(Path(relative).parts) - 1]
        if WORK in path.parents:
            where = "built here"
        elif INPUTS.exists() and (INPUTS in root.parents or root == INPUTS):
            where = root.relative_to(INPUTS).as_posix()
        else:
            where = str(root)
        print(f"  {label:<14}{size_mb(path):>9.0f} MB   {where}")
print("code")
PROJECT = get_code()
sys.path.insert(0, str(PROJECT / "src"))
os.chdir(PROJECT)
print(f"  {PROJECT}")

print("\nsecrets")
load_secrets(PROJECT)

FOUND, MISSING = locate(PROJECT)
if MISSING:
    raise SystemExit(
        "Notebook 1's output is not attached, and nothing is built in this notebook.\n"
        f"  missing:  {', '.join(MISSING)}\n"
        f"  attached: {attached_inputs() or 'nothing'}\n\n"
        "  Add Input > Notebook Output > NL2SQL 1 Setup\n"
        "  https://www.kaggle.com/code/kirazul/nl2sql-1-setup"
    )

configure(FOUND)
print("\nartefacts")
show(FOUND)

---

## 2. The cloud provider

Three of the four architectures call a cloud model. This checks it answers before
running twelve questions against it, so a missing key shows up here as one line
rather than as four empty rows in the results table.

In [ ]:
from hybridsql.providers import cloud

targets = cloud.chain()
for target in targets:
    print(f"  {target.name}")

if not targets:
    raise SystemExit(
        "No cloud provider is configured. Three of the four architectures call one,\n"
        "and without a key they fail instantly with no tokens and no rows.\n"
        "Set GROQ_API_KEY under Add-ons > Secrets, attached to this notebook,\n"
        "then run the setup cell again."
    )
print(f"\n  {len(targets)} target(s), tried in this order")

---

## 3. Four architectures

Each one answers the same question. They differ in who writes the SQL, who writes
the answer, and what crosses the network to make that happen.

| | Writes the SQL | Writes the answer | What is sent | Values sent |
|---|---|---|---|---|
| **Hybrid** | cloud, from symbols | local | schema and a masked question | 0 |
| **Hybrid Opaque** | cloud, from labels | local | labels only, no business word | 0 |
| **Full Cloud** | cloud, raw question | cloud | the question and every row | all |
| **Full Local** | local 1.7 B | local | nothing | 0 |

They are built as state machines that share every stage they have in common, so a
fix to execution is a fix in all four. The diagrams below are generated from the
compiled graphs, which means they cannot describe an architecture that is not the
one running.

In [ ]:
from hybridsql.graph import ARMS, run
from hybridsql.graph.state import public

QUESTIONS = [
    "How many patients received aspirin?",
    "What is the average age of patients admitted to the MICU?",
    "How many female patients were discharged alive?",
]

results = []


def ask(question, arm, write=False):
    """Run one question through one architecture and print every stage of it."""
    r = public(run(question, arm=arm, write=write))
    results.append(r)

    print(f"  question    {question}")
    if r["masked_question"]:
        print(f"  sent        {r['masked_question']}")
    if r["opaque"].get("question"):
        print(f"  relabelled  {r['opaque']['question']}")
    if r["sql"]:
        print(f"  sql         {' '.join(r['sql'].split())}")
    if r["success"]:
        print(f"  rows        {r['row_count']}")
    else:
        print(f"  failed      {r['failed_stage']}: {r['failure_reason']}")
    if r["answer"]:
        print(f"  answer      {' '.join(r['answer'].split())[:180]}")
    print(f"  cost        {sum(r['ms'].values()):.0f} ms, {r['cloud_tokens']} cloud tokens, "
          f"{r['egress_chars']} chars sent, {r['egress_values']} values sent\n")
    return r

In [ ]:
from hybridsql import graph
from IPython.display import Markdown, display

for arm in ARMS:
    display(Markdown(f"**{arm}**\n\n```mermaid\n{graph.mermaid(arm)}\n```"))

---

## 4. Hybrid

Hybrid gives the cloud model the schema and a sentence with holes in it. The
model writes `WHERE drugname = :v1`. The value is bound here, through the SQLite
driver, so the query text and the value never meet in a single string.

**Modules:** `pipeline/understand.py`, `pipeline/anonymize.py`,
`pipeline/generate.py`, `security/egress_gate.py`, `providers/cloud.py`,
`db/connection.py`, `pipeline/answer.py`

**This is the whole prompt that leaves.**

In [ ]:
from hybridsql.pipeline.understand import understand
from hybridsql.pipeline.anonymize import anonymize
from hybridsql.pipeline import generate as gen

u = understand(QUESTIONS[0])
a = anonymize(u)

for message in gen.build_messages(u, a):
    print(f"--- {message['role']} " + "-" * 56)
    print(message["content"][:850])
    print()
print(f"kept here: {a.mapping}")

**Running it.**

In [ ]:
for q in QUESTIONS:
    ask(q, "hybrid")

### The answer is written here

The rows never left. Turning them into a sentence is the local model's job, and
it is what lets the results of a protected architecture stay on this machine. The
first call loads 1.1 GB of weights, so it is slow once and quick afterwards.

In [ ]:
written = public(run(QUESTIONS[0], arm="hybrid", write=True))

print(f"  question           {QUESTIONS[0]}")
print(f"  sql written by     {written['sql_author']}")
print(f"  rows               {written['row_count']}, none of which left this process")
print(f"  answer written by  {written['answer_author']}")
print(f"\n  {written['answer']}")

---

## 5. Hybrid Opaque

Hybrid still sends the schema, and 31 table names with 391 column names describe
a business even when no row does. Hybrid Opaque replaces those too. `medication`
becomes `t3`, `drugname` becomes `c7`, and the labels are drawn again on every
request.

Stripping schema names normally wrecks text-to-SQL, because the names carry the
meaning. It works here because the meaning was already resolved locally. The
index has established which column the value belongs to, so the prompt can state
`:v1 is a value of c7` and leave the model a mechanical job: follow the foreign
keys, place the aggregate. The SQL that comes back is translated to real names
here.

**Modules:** Hybrid's, plus `pipeline/opaque.py`

**What Hybrid sends, and what Opaque sends instead.**

In [ ]:
view = gen.build_opaque(u, a).view()

print(f"  asked       {QUESTIONS[0]}")
print(f"  hybrid      {a.masked_question}")
print(f"  opaque      {view['question']}")
print(f"\n  {view['parameters'].strip()}")
print(f"\n  schema: {view['tables']} tables, {view['columns']} columns, every name a label")
print(view["ddl"][:420])

**The dictionary that stays here.** Read it right to left: these names never left.

In [ ]:
for alias, real in view["labels"].items():
    print(f"  {alias:<6}{real}")

**Running it.** The line to watch is `relabelled`. That is what the model received.

In [ ]:
for q in QUESTIONS:
    ask(q, "hybrid_opaque")

---

## 6. Full Cloud

Full Cloud is the baseline. There is no masking stage, and its absence is the
architecture: the question leaves as typed, the model writes the SQL, and then
the rows are sent back to the model so it can write the answer. Every cell of
every row crosses the network.

It is here to be measured, not recommended. The egress gate is bypassed
explicitly, `PRIVACY_MODE=strict` refuses that outright, and the audit journal
records every bypass.

**Modules:** `pipeline/generate.py`, `providers/cloud.py`, `db/connection.py`

In [ ]:
for q in QUESTIONS:
    ask(q, "full_cloud")

**The journal.** Every bypass is written down.

In [ ]:
from hybridsql.security import audit

lines = audit.read()
for key, value in audit.leak_rate().items():
    print(f"  {key:<20}{value}")
print(f"  {'bypassed':<20}{sum(1 for line in lines if line.get('bypassed'))}")

---

## 7. Full Local

Full Local sends nothing. The 1.7 B model writes the SQL itself, on two CPU
cores, from the same schema the cloud model would have received.

It is the other end of the range. The gap between it and Full Cloud is what makes
the two middle architectures worth building: if a 1.7 B model wrote SQL as well
as a 120 B model, there would be nothing to protect and nothing to argue about.

**Modules:** `pipeline/understand.py`, `providers/local_model.py`,
`db/connection.py`, `pipeline/answer.py`

In [ ]:
for q in QUESTIONS:
    ask(q, "full_local")

---

## 8. The comparison

`ran` counts queries that executed, not answers that were right. Accuracy is a
separate evaluation, in `scripts/evaluate_pipeline.py`.

The column that decides is **values sent**.

In [ ]:
from collections import defaultdict

rows = defaultdict(lambda: {"n": 0, "ok": 0, "ms": 0.0, "values": 0, "chars": 0, "tokens": 0})
for r in results:
    e = rows[r["arm"]]
    e["n"] += 1
    e["ok"] += int(r["success"])
    e["ms"] += sum(r["ms"].values())
    e["values"] += r["egress_values"]
    e["chars"] += r["egress_chars"]
    e["tokens"] += r["cloud_tokens"]

head = f"{'architecture':<16}{'ran':>7}{'avg ms':>9}{'values sent':>13}{'chars sent':>12}{'tokens':>8}"
print(head)
print("-" * len(head))
for arm in ARMS:
    e = rows[arm]
    if e["n"]:
        print(f"{arm:<16}{str(e['ok']) + '/' + str(e['n']):>7}{e['ms'] / e['n']:>9.0f}"
              f"{e['values']:>13}{e['chars']:>12}{e['tokens']:>8}")

failed = [r for r in results if not r["success"]]
if failed:
    print("\nwhat failed")
    for r in failed:
        print(f"  {r['arm']:<16}{r['failed_stage']}: {r['failure_reason'][:70]}")

Hybrid and Hybrid Opaque send zero values. Full Cloud sends every one it touches.
Full Local sends nothing at all and pays for it in the `ran` column.

---

The same pipeline in one run, then the live service.

**Next:** [4. Run All](https://www.kaggle.com/code/kirazul/nl2sql-4-run-all)